In [1]:

import os
import requests
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import Tool
from langchain.agents import create_agent



if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter Tavily API Key: ")



MODEL = "gemini-3.5-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)


def run_qa(query):

    response = llm.invoke(
        f"Answer the following question clearly:\n{query}"
    )

    return response.content


qa_tool = Tool(
    name="simple_qa",
    func=run_qa,
    description=(
        "Answers general factual and conceptual questions. "
        "Use this tool for explanations and knowledge-based questions."
    )
)



def run_summary(text):

    response = llm.invoke(
        f"Summarize the following text clearly and concisely:\n\n{text}"
    )

    return response.content


summary_tool = Tool(
    name="summarizer",
    func=run_summary,
    description=(
        "Summarizes paragraphs, articles, and long text. "
        "Use this tool when the user requests a summary."
    )
)



def run_web_search(query):

    url = "https://api.tavily.com/search"

    payload = {
        "api_key": os.environ["TAVILY_API_KEY"],
        "query": query,
        "max_results": 3
    }

    response = requests.post(
        url,
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    results = response.json().get("results", [])

    if not results:
        return "No search results found."

    formatted_results = []

    for result in results:

        formatted_results.append(
            f"Title: {result.get('title', '')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Content: {result.get('content', '')}"
        )

    return "\n\n".join(formatted_results)


search_tool = Tool(
    name="web_search",
    func=run_web_search,
    description=(
        "Searches the internet for latest news, current events, "
        "recent updates, and up-to-date information."
    )
)



tools = [
    qa_tool,
    summary_tool,
    search_tool
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful AI assistant with access to three tools.

    1. simple_qa:
       Answers factual and conceptual questions.

    2. summarizer:
       Summarizes long paragraphs and text.

    3. web_search:
       Searches the internet for current information.

    Instructions:
    - Understand the user's query.
    - Select the most appropriate tool.
    - Use web_search for latest or current information.
    - Use summarizer when the user asks to summarize text.
    - Use simple_qa for factual and conceptual questions.
    - Use the tool results to generate a clear final answer.
    - Do not claim that you searched the web unless web_search was used.
    """
)



queries = [

    "What is LangGraph in LangChain?",

    "Summarize this: LangChain is a framework to build LLM applications using prompts, memory, tools, and agents.",

    "Find the latest news about OpenAI GPT-4o."

]


for query in queries:

    print("\n" + "=" * 60)
    print("USER QUERY:", query)
    print("=" * 60)

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": query}
        ]
    })

    print("\nAGENT RESPONSE:")

    print(response["messages"][-1].content)


Enter Gemini API Key: ··········
Enter Tavily API Key: ··········

USER QUERY: What is LangGraph in LangChain?

AGENT RESPONSE:
[{'type': 'text', 'text': '**LangGraph** is a library created by the LangChain team to help developers build **complex, stateful, and multi-actor LLM applications** using graph-based architectures.\n\n---\n\n### Why LangGraph? (Chains vs. Graphs)\n\n* **Standard LangChain (Chains):** Operates primarily in a linear, single-direction flow (Directed Acyclic Graph / DAG). This is ideal for straightforward tasks like basic Q&A, document retrieval, or simple summarization, but it struggles with complex, iterative decision-making.\n* **LangGraph (Graphs):** Enables **cyclic workflows** (loops). An AI agent can perform an action, evaluate its output, correct errors, and loop back to try again until a task is successfully completed.\n\n---\n\n### Core Concepts\n\nLangGraph models applications using three main components:\n\n1. **State:** A shared data structure passed 